# 01 — SECOM Exploratory Data Analysis

Descriptive EDA on the P1-001 cleaned SECOM outputs. No feature screening,
no SPC, no modeling. All inputs come from `src/config.yaml` paths; key
numbers are written to `reports/secom_eda_summary.json` and figures to
`reports/figures/secom_*.png`.

In [1]:
from pathlib import Path

import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import yaml

candidate = Path.cwd()
while not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
ROOT = candidate
sys.path.insert(0, str(ROOT))

from src.config_loader import load_config

config = load_config()
secom = config["secom"]["outputs"]
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

cleaning_summary = json.loads((ROOT / secom["cleaning_summary"]).read_text(encoding="utf-8"))
profile = pd.read_csv(ROOT / secom["missingness_profile"])
cleaned = pd.read_csv(ROOT / secom["cleaned_data"], parse_dates=["timestamp"])
sensor_cols = [c for c in cleaned.columns if c.startswith("sensor_")]
print("cleaned shape:", cleaned.shape, "| sensors:", len(sensor_cols))

cleaned shape: (1567, 450) | sensors: 446


## 1. Shape, target distribution, class imbalance

In [2]:
shape = {"rows": int(cleaned.shape[0]), "columns": int(cleaned.shape[1]),
         "sensor_columns": len(sensor_cols)}
target_counts = cleaned["target_label"].value_counts().to_dict()
fail_rate = float((cleaned["target"] == 1).mean())
imbalance_ratio = float((cleaned["target"] == 0).sum() / max((cleaned["target"] == 1).sum(), 1))
print(shape)
print(target_counts)
print(f"fail_rate={fail_rate:.4f} imbalance_ratio(majority:minority)={imbalance_ratio:.2f}")

{'rows': 1567, 'columns': 450, 'sensor_columns': 446}
{'pass': 1463, 'fail': 104}
fail_rate=0.0664 imbalance_ratio(majority:minority)=14.07


## 2. Timestamp range and yield trend over time

In [3]:
ts_min, ts_max = cleaned["timestamp"].min(), cleaned["timestamp"].max()
span_days = (ts_max - ts_min).days
daily = cleaned.assign(date=cleaned["timestamp"].dt.date).groupby("date").agg(
    lots=("target", "size"), fails=("target", "sum"))
daily["fail_rate"] = daily["fails"] / daily["lots"]
print(f"range: {ts_min} -> {ts_max} ({span_days} days, {len(daily)} active dates)")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].bar(daily.index.astype(str), daily["lots"])
axes[0].set_ylabel("lots per day")
axes[1].plot(daily.index.astype(str), daily["fail_rate"], marker="o", ms=3)
axes[1].set_ylabel("daily fail rate")
axes[1].tick_params(axis="x", rotation=90, labelsize=6)
fig.suptitle("SECOM daily volume and fail rate (yield trend)")
fig.tight_layout()
fig.savefig(FIG_DIR / "secom_daily_volume_failrate.png", dpi=120)
plt.close(fig)
print("worst day:", daily["fail_rate"].idxmax(), float(daily["fail_rate"].max()))

range: 2008-07-19 11:55:00 -> 2008-10-17 06:07:00 (89 days, 86 active dates)


worst day: 2008-07-25 1.0


## 3. Missingness: distribution, removed vs retained

In [4]:
removed_null = cleaning_summary["removed_high_null_sensors"]["count"]
removed_zero = cleaning_summary["removed_zero_variance_sensors"]["count"]
retained = cleaning_summary["final_sensor_count"]
original = cleaning_summary["original_sensor_count"]
print(f"original={original} removed_high_null={removed_null} "
      f"removed_zero_var={removed_zero} retained={retained}")
assert removed_null + removed_zero + retained == original

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(profile["missing_fraction"], bins=40)
ax.set_xlabel("missing fraction per sensor")
ax.set_ylabel("sensor count")
ax.set_title("SECOM missingness distribution (591 original sensors)")
fig.tight_layout()
fig.savefig(FIG_DIR / "secom_missingness_hist.png", dpi=120)
plt.close(fig)
print(profile["missing_fraction"].describe().to_dict())

original=590 removed_high_null=28 removed_zero_var=116 retained=446
{'count': 590.0, 'mean': 0.045375488085847356, 'std': 0.1543398625094035, 'min': 0.0, '25%': 0.00127632418634, '50%': 0.00382897255903, '75%': 0.00574345883854, 'max': 0.911933631142}


## 4. Retained sensor distributions and variance

In [5]:
variances = cleaned[sensor_cols].var()
print("retained variance summary:", variances.describe().to_dict())
top_var = variances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(variances, bins=40)
ax.set_xscale("log")
ax.set_xlabel("variance (log scale)")
ax.set_ylabel("sensor count")
ax.set_title("Variance distribution across retained sensors")
fig.tight_layout()
fig.savefig(FIG_DIR / "secom_variance_hist.png", dpi=120)
plt.close(fig)
print("top-15 variance sensors:", top_var.index.tolist())

retained variance summary: {'count': 446.0, 'mean': 202389.464356313, 'std': 2297649.7111250805, 'min': 1.0901024983984624e-08, '25%': 0.0024234237678625076, '50%': 1.2938520871307952, '75%': 94.24831021779868, 'max': 42949270.79576657}


top-15 variance sensors: ['sensor_162', 'sensor_161', 'sensor_297', 'sensor_024', 'sensor_296', 'sensor_023', 'sensor_159', 'sensor_021', 'sensor_160', 'sensor_204', 'sensor_140', 'sensor_294', 'sensor_003', 'sensor_225', 'sensor_090']


## 5. Obvious outliers (IQR rule per sensor)

In [6]:
q1 = cleaned[sensor_cols].quantile(0.25)
q3 = cleaned[sensor_cols].quantile(0.75)
iqr = (q3 - q1).replace(0, float("nan"))
outlier_frac = ((cleaned[sensor_cols] < (q1 - 1.5 * iqr)) |
                (cleaned[sensor_cols] > (q3 + 1.5 * iqr))).mean().sort_values(ascending=False)
print("top-15 outlier-fraction sensors:")
print(outlier_frac.head(15).to_dict())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(outlier_frac.dropna(), bins=40)
ax.set_xlabel("fraction of rows outside 1.5*IQR")
ax.set_ylabel("sensor count")
ax.set_title("Outlier-rate distribution across retained sensors")
fig.tight_layout()
fig.savefig(FIG_DIR / "secom_outlier_hist.png", dpi=120)
plt.close(fig)

top-15 outlier-fraction sensors:
{'sensor_031': 0.22590938098276964, 'sensor_040': 0.21059349074664965, 'sensor_544': 0.1525207402680281, 'sensor_129': 0.15188257817485643, 'sensor_312': 0.14932992980216975, 'sensor_195': 0.14869176770899808, 'sensor_177': 0.1467772814294831, 'sensor_448': 0.1467772814294831, 'sensor_467': 0.1391193363114231, 'sensor_562': 0.1372048500319081, 'sensor_251': 0.12827058072750477, 'sensor_523': 0.12699425654116145, 'sensor_059': 0.12507977026164646, 'sensor_041': 0.1244416081684748, 'sensor_023': 0.12380344607530312}


## 6. Correlations between high-variance sensors

In [7]:
top10 = variances.sort_values(ascending=False).head(10).index.tolist()
corr = cleaned[top10].corr()
high_pairs = [(top10[i], top10[j], float(corr.iloc[i, j]))
              for i in range(len(top10)) for j in range(i + 1, len(top10))
              if abs(float(corr.iloc[i, j])) > 0.9]
print(f"{len(high_pairs)} pairs with |r| > 0.9")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.matshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(top10)), top10, rotation=90, fontsize=7)
ax.set_yticks(range(len(top10)), top10, fontsize=7)
ax.set_title("Correlation: top-10 variance sensors")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIG_DIR / "secom_top10_corr.png", dpi=120)
plt.close(fig)

2 pairs with |r| > 0.9


## 7. Pass vs fail sensor contrasts (descriptive only, no screening)

In [8]:
contrast_sensors = variances.sort_values(ascending=False).head(8).index.tolist()
passed = cleaned.loc[cleaned["target"] == 0, contrast_sensors]
failed = cleaned.loc[cleaned["target"] == 1, contrast_sensors]
pooled = (((passed.std() ** 2 * (len(passed) - 1) + failed.std() ** 2 * (len(failed) - 1))
           / max(len(passed) + len(failed) - 2, 1)) ** 0.5).replace(0, float("nan"))
smd = ((failed.mean() - passed.mean()) / pooled).sort_values(key=abs, ascending=False)
contrast = {s: {"mean_pass": float(passed[s].mean()), "mean_fail": float(failed[s].mean()),
                "std_mean_diff": float(smd[s])} for s in contrast_sensors}
for sensor, values in contrast.items():
    print(sensor, {k: round(v, 4) for k, v in values.items()})

sensor_162 {'mean_pass': 4807.2941, 'mean_fail': 4653.233, 'std_mean_diff': -0.0235}
sensor_161 {'mean_pass': 4088.8953, 'mean_fail': 3753.9417, 'std_mean_diff': -0.079}
sensor_297 {'mean_pass': 2346.1919, 'mean_fail': 2295.0653, 'std_mean_diff': -0.0158}
sensor_024 {'mean_pass': -284.4415, 'mean_fail': -499.5396, 'std_mean_diff': -0.0741}
sensor_296 {'mean_pass': 1887.5381, 'mean_fail': 1761.2783, 'std_mean_diff': -0.0639}
sensor_023 {'mean_pass': -3811.0356, 'mean_fail': -3739.0785, 'std_mean_diff': 0.0521}
sensor_159 {'mean_pass': 862.2756, 'mean_fail': 1172.3107, 'std_mean_diff': 0.3161}
sensor_021 {'mean_pass': -5636.4376, 'mean_fail': -5362.2743, 'std_mean_diff': 0.4397}


## 8. Reconciliation and evidence export

In [9]:
assert cleaned.shape[0] == cleaning_summary["original_row_count"] == 1567
assert (cleaned["target_label"] == "pass").sum() == cleaning_summary["target_distribution"]["pass"]
assert (cleaned["target_label"] == "fail").sum() == cleaning_summary["target_distribution"]["fail"]
assert len(sensor_cols) == retained

eda_summary = {
    "shape": shape,
    "target_counts": {k: int(v) for k, v in target_counts.items()},
    "fail_rate": fail_rate,
    "imbalance_ratio": imbalance_ratio,
    "timestamp_min": str(ts_min),
    "timestamp_max": str(ts_max),
    "span_days": span_days,
    "removed_high_null": removed_null,
    "removed_zero_variance": removed_zero,
    "retained_sensors": retained,
    "missing_fraction_summary": {k: float(v) for k, v in profile["missing_fraction"].describe().to_dict().items()},
    "variance_summary": {k: float(v) for k, v in variances.describe().to_dict().items()},
    "top_variance_sensors": top_var.index.tolist(),
    "top_outlier_fraction": {k: float(v) for k, v in outlier_frac.head(15).items()},
    "high_corr_pairs_gt_0_9": [{"a": a, "b": b, "r": r} for a, b, r in high_pairs],
    "pass_fail_contrast_top8": contrast,
    "daily_fail_rate": {str(k): float(v) for k, v in daily["fail_rate"].items()},
    "figures": ["secom_daily_volume_failrate.png", "secom_missingness_hist.png",
                "secom_variance_hist.png", "secom_outlier_hist.png", "secom_top10_corr.png"],
}
out = ROOT / "reports" / "secom_eda_summary.json"
out.write_text(json.dumps(eda_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("wrote", out)

wrote C:\Users\luqma\OneDrive\Desktop\Data Analyst Project\Production Yield Loss and OEE Early\reports\secom_eda_summary.json


## Findings

- SECOM: 1,567 lots; fail minority class (~6–7%, verify number above).
- Heavy sensor missingness drove most removals; retained sensors span many
  orders of variance — screening in a later phase must be variance-aware.
- No SPC, screening, or modeling performed here by design.